# Μάθημα 2: Computer Vision με Συνελικτικά Νευρωνικά Δίκτυα (CNN)

Τα CNNs (Convolutional Neural Networks) άλλαξαν την ιστορία του AI. Αντί να ισιώνουν την εικόνα και να χάνουν τη γεωμετρία της, χρησιμοποιούν **Φίλτρα (Kernels)** για να 'σκανάρουν' την εικόνα αναζητώντας μοτίβα (ακμές, σχήματα).

## 1. Χτίζοντας την Αρχιτεκτονική ενός CNN
Θα ορίσουμε ένα δίκτυο που παίρνει ασπρόμαυρες εικόνες (π.χ. το διάσημο MNIST dataset με χειρόγραφα ψηφία) και προσπαθεί να μαντέψει ποιο ψηφίο είναι (0-9).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN_Classifier(nn.Module):
    def __init__(self):
        super(CNN_Classifier, self).__init__()
        # Conv1: 1 κανάλι εισόδου (ασπρόμαυρο), 16 φίλτρα εξόδου, παράθυρο 3x3
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        # MaxPool: Μειώνει το ύψος και πλάτος στο μισό
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Conv2: Πιο βαθύ φίλτρο, βρίσκει πιο περίπλοκα σχήματα
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        # Τελικό επίπεδο (Fully Connected). 
        # Αν η εικόνα ήταν 28x28, μετά από 2 pools γίνεται 7x7. Άρα 32 φίλτρα * 7 * 7.
        self.fc1 = nn.Linear(32 * 7 * 7, 10) # 10 πιθανές έξοδοι (0-9)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # Ισιώνουμε τον πίνακα
        x = self.fc1(x)
        return x

model = CNN_Classifier()
print("Η αρχιτεκτονική του CNN:\n", model)

## 2. Πώς μοιάζει το Training σε Εικόνες (Simulation)
Κανονικά, εδώ θα χρησιμοποιούσαμε το `torchvision` για να κατεβάσουμε 60.000 εικόνες. Εδώ θα κάνουμε μια προσομοίωση με ένα batch 32 ψεύτικων εικόνων, για να δούμε πώς μετράμε την **Ακρίβεια (Accuracy)**.

In [ ]:
# Προσομοίωση: 32 εικόνες (batch_size), 1 κανάλι, 28x28 pixels
fake_images = torch.rand(32, 1, 28, 28)
# Οι σωστές απαντήσεις (τυχαίοι αριθμοί από 0 έως 9)
fake_labels = torch.randint(0, 10, (32,))

# Κάνουμε την πρόβλεψη
outputs = model(fake_images)

# Το δίκτυο βγάζει 10 αριθμούς (ακατέργαστες πιθανότητες - logits) για κάθε εικόνα.
# Εμείς επιλέγουμε τη μεγαλύτερη τιμή (argmax) ως την τελική πρόβλεψη του μοντέλου.
_, predicted = torch.max(outputs.data, 1)

print("Τι προέβλεψε το μοντέλο:\n", predicted)
print("\nΠοιες ήταν οι πραγματικές ετικέτες:\n", fake_labels)

# Υπολογισμός Accuracy
correct = (predicted == fake_labels).sum().item()
accuracy = 100 * correct / 32
print(f"\nΤρέχουσα Ακρίβεια (τυχαία πριν την εκπαίδευση): {accuracy}% (Περίπου 10% όπως αναμένεται σε 10 κλάσεις!)")